In [1]:
# ── From PyPI (recommended)
!pip install cvxeda                                # latest release
#!pip install "cvxeda==1.0.0"                       # or a specific version

  Using cached cvxeda-1.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached cvxopt-1.3.3.tar.gz (8.9 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached cvxeda-1.1.0-py3-none-any.whl (19 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for cvxopt (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [62 lines of output]
      /private/var/folders/zc/kzlmzxqx1bx13blggjybvfx1xkbyfl/T/pip-build-env-lv9007dt/overlay/lib/python3.11/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
 

In [ ]:
import pandas as pd
import cvxeda

In [ ]:
df = pd.read_csv("../../Data/Processed/MatLab_EDA_For_R.csv")

In [ ]:
df.head()

In [ ]:
test = df.iloc[:, 0]
test.head()

In [ ]:

r, p, t, l, d, e, obj = cvxeda.cvxEDA(test, 1.0 / 4)

In [ ]:
r.shape

In [ ]:
test.shape

In [ ]:
t.shape

In [ ]:
df.shape

In [ ]:
# Initialize an empty list to store the 'r' results for each column
processed_r_columns = []

# Iterate over each column name in the DataFrame df
for col_name in df.columns:
  # Select the current column by its name
  current_column_series = df[col_name]

  # Apply cvxEDA to the current column Series
  # Assuming 'r' is the primary output of interest to collect for each column
  r, _, _, _, _, _, _ = cvxeda.cvxEDA(current_column_series, 1.0 / 4)

  # Create a pandas Series from the 'r' output, using the original index for alignment
  # and a meaningful column name (e.g., 'r_SUD_003_01_EDA')
  r_series = pd.Series(r, index=df.index, name=f'Phasic_{col_name}')

  # Add this processed Series to our list
  processed_r_columns.append(r_series)

# After processing all columns, concatenate the list of Series into a new DataFrame
# This will create a DataFrame where each column is the 'r' component for an original df column
CVX_Phasic = pd.concat(processed_r_columns, axis=1)

# You might want to display the head of the new DataFrame to check the result
CVX_Phasic.head()

In [ ]:
# Initialize an empty list to store the 'r' results for each column
processed_t_columns = []

# Iterate over each column name in the DataFrame df
for col_name in df.columns:
  # Select the current column by its name
  current_column_series = df[col_name]

  # Apply cvxEDA to the current column Series
  # Assuming 'r' is the primary output of interest to collect for each column
  _, _, t, _, _, _, _ = cvxeda.cvxEDA(current_column_series, 1.0 / 4)

  # Create a pandas Series from the 'r' output, using the original index for alignment
  # and a meaningful column name (e.g., 'r_SUD_003_01_EDA')
  t_series = pd.Series(t, index=df.index, name=f'tonic_{col_name}')

  # Add this processed Series to our list
  processed_t_columns.append(t_series)

# After processing all columns, concatenate the list of Series into a new DataFrame
# This will create a DataFrame where each column is the 'r' component for an original df column
CVX_Tonic = pd.concat(processed_t_columns, axis=1)

CVX_Tonic.head()

In [ ]:
df.to_csv('CVX_Tonic.csv', index=False)

In [ ]:
%load_ext rpy2.ipython


In [ ]:
%%R -i df,CVX_Tonic,CVX_Phasic -o result

assign_condition_by_row <- function(row_index) {
  if (row_index >= 1 && row_index <= 239) return("neutral")
  if (row_index >= 240 && row_index <= 539) return("black")
  if (row_index >= 540 && row_index <= 779) return("positive")
  if (row_index >= 780 && row_index <= 1075) return("black")
  if (row_index >= 1076 && row_index <= 1300) return("negative")
  return(NA)
}

CVX_Phasic$condition <- sapply(1:nrow(CVX_Phasic), assign_condition_by_row)
CVX_Phasic <- CVX_Phasic[!is.na(CVX_Phasic$condition), ]
CVX_Phasic$condition <- factor(CVX_Phasic$condition,
                              levels = c("neutral", "positive", "negative", "black"))

reshape_by_condition <- function(df_input, window_size = 10, overlap = 0) {

  chunk_size <- window_size * 4
  colnames_data <- colnames(df_input)[colnames(df_input) != "condition"]

  rle_cond <- rle(as.character(df_input$condition))
  block_ends <- cumsum(rle_cond$lengths)
  block_starts <- c(1, head(block_ends, -1) + 1)

  chunk_list <- mapply(function(start, end, cond) {
    row_indices <- start:end
    n_rows <- length(row_indices)
    n_chunks <- ceiling(n_rows / chunk_size)

    padded <- c(row_indices, rep(NA, n_chunks * chunk_size - n_rows))
    chunks <- split(padded, ceiling(seq_along(padded) / chunk_size))

    list(chunks = chunks, condition = cond, n_chunks = n_chunks)
  }, block_starts, block_ends, rle_cond$values, SIMPLIFY = FALSE)

  if (overlap > 0) {
    for (i in seq_len(length(chunk_list) - 1)) {
      current_block <- chunk_list[[i]]
      next_block_start <- block_starts[i + 1]

      overlap_indices <- next_block_start:min(next_block_start + overlap - 1, block_ends[i + 1])

      last_chunk <- current_block$chunks[[current_block$n_chunks]]
      real_indices <- last_chunk[!is.na(last_chunk)]

      combined <- c(real_indices, overlap_indices)

      if (length(combined) <= chunk_size) {
        last_chunk <- c(combined, rep(NA, chunk_size - length(combined)))
      } else {
        last_chunk <- combined
      }

      chunk_list[[i]]$chunks[[current_block$n_chunks]] <- last_chunk
    }
  }

  all_chunks <- do.call(c, lapply(chunk_list, function(x) x[["chunks"]]))
  all_conditions <- unlist(lapply(chunk_list, function(b) rep(b$condition, b$n_chunks)))

  n_cols2 <- length(all_chunks)

  df_reshaped_list2 <- lapply(colnames_data, function(col_name) {
    vals <- df_input[[col_name]]
    mat <- sapply(all_chunks, function(idx) {
      n_real <- sum(!is.na(idx))
      out <- rep(NA, max(chunk_size, length(idx)))
      out[seq_along(idx)[!is.na(idx)]] <- vals[idx[!is.na(idx)]]
      out[seq_len(chunk_size)]
    })
    as.data.frame(mat)
  })

  df_reshaped2 <- do.call(cbind, df_reshaped_list2)

  if (n_cols2 > 0) {
    new_colnames2 <- unlist(lapply(colnames_data, function(col_name) {
      paste0(col_name, "_", seq(window_size, n_cols2 * window_size, by = window_size), "s")
    }))
    colnames(df_reshaped2) <- new_colnames2
  } else {
    message("Warning: n_cols2 is 0, no column renaming will occur.")
  }

  df_transposed2 <- as.data.frame(t(df_reshaped2))
  df_transposed2$condition <- factor(all_conditions, levels = c("neutral", "positive", "negative", "black"))

  chunk_row_indices <- sapply(all_chunks, function(idx) {
    paste(idx[!is.na(idx)], collapse = ", ")
  })
  df_transposed2$source_rows <- chunk_row_indices

  return(df_transposed2)
}

result <- reshape_by_condition(CVX_Phasic)

In [ ]:
result.head()

In [ ]:
result.to_csv('Phasic_Reshaped.csv', index=True)

In [ ]:
numeric_result = result.drop(columns=['condition', 'source_rows'])
phasic_desc = pd.DataFrame({
    'Mean_Phasic': numeric_result.mean(axis=1),
    'Max_Phasic':  numeric_result.max(axis=1),
    'Min_Phasic':  numeric_result.min(axis=1),
    'SD_Phasic':   numeric_result.std(axis=1)
})

print(phasic_desc)

In [ ]:
Tonic_Reshaped = pd.read_csv("Tonic_Reshaped.csv")

In [ ]:
numeric_result = result.drop(columns=['condition', 'source_rows'])
Tonic_desc = pd.DataFrame({
    'Mean_Tonic': numeric_result.mean(axis=1),
    'Max_Tonic':  numeric_result.max(axis=1),
    'Min_Tonic':  numeric_result.min(axis=1),
    'SD_Tonic':   numeric_result.std(axis=1)
})

print(Tonic_desc)

In [ ]:
Final_Dataframe = pd.read_csv("Raw_Features.csv")
Final_Dataframe.head()

In [ ]:
Tonic_desc_for_merge = Tonic_desc.copy()
Tonic_desc_for_merge['X'] = Tonic_desc_for_merge.index.str.replace('tonic_', '')
result = pd.merge(Final_Dataframe, Tonic_desc_for_merge, on='X', how='left')

In [ ]:
result.head()

In [ ]:
Phasic_desc_for_merge = phasic_desc.copy()
Phasic_desc_for_merge['X'] = Phasic_desc_for_merge.index.str.replace('Phasic_', '')
result = pd.merge(result, Phasic_desc_for_merge, on='X', how='left')
result.head()

In [ ]:
result.to_csv('Final_Result.csv', index=False)